# Eugene's Integral Computations 

## Import Packages

In [2]:
using StaticArrays 
using LinearAlgebra # provides the function norm 
using HCubature # provides adaptive numerical integration 

using Test # provides testing functionality 
using Plots 

const Point3D = SVector{3,Float64};

## Section 1: Introduction 

We describe here how we wish to proceed. 

### Bottom-up Approach 

1. implement the testing by comparing symbolic and numerical evaluation of the integrals; 

### Top-down Approach 

1. assume two prism to be given;
2. reduce 6D integrals to sum of 4D integrals (how many exactly?) using Euler's theorem for homogeneous functions;
3. evaluate the 4D integrals numerically by some adaptive quadrature;
     

## Section 2: Edges and Triangles as Function-Like Object 
See [Function-like-objects in the Julia Manual](https://docs.julialang.org/en/v1/manual/methods/#Function-like-objects).

### A tutorial example 

In [69]:
# define polynomial as vector of coefficients
struct Polynomial{Float64}
    coeffs::Vector{Float64}
end

# add function to evaluate the polynomial to the type definition   
function (p::Polynomial)(x)
    # use data stored in member function 
    v = p.coeffs[end]
    for i = (length(p.coeffs)-1):-1:1
         v = v*x + p.coeffs[i]
    end
    return v
end

# define in which value the polynomial should be evaluated in case that no arguments are passed 
(p::Polynomial)() = p(0)

In [70]:
p = Polynomial([1.,10.,100.])

Polynomial{Float64}([1.0, 10.0, 100.0])

In [71]:
dump(p)

Polynomial{Float64}
  coeffs: Array{Float64}((3,)) [1.0, 10.0, 100.0]


In [72]:
p(2.)

421.0

In [73]:
function do_integrate(poly)
    value = hcubature(x -> poly(x[1]) , (0,), (1,)) 
    return value
end 

do_integrate (generic function with 1 method)

### An edge data structure 

In [74]:
struct Foo 
    p1::Float64
    p2::Float64 
end 

function (f::Foo)(x)
    return [f.p1*x,f.p2*x]  
end

In [75]:
myfoo = Foo(2.,3.)

Foo(2.0, 3.0)

In [76]:
myfoo(5.)[1]

10.0

In [9]:
struct Edge3D 
    len::Float64
    r1::Point3D
    r2::Point3D
    dir::Point3D 
end 

function (edge3D::Edge3D)(r)
    e = edge3D.dir / edge3D.len  
    return [dot(edge3D.r2-r,e), dot(r-edge3D.r1,e)] 
end

In [13]:
r1 = Point3D(1.,0.,0.,); r2 = Point3D(2.,0.,0.); len = norm(r2-r1)
s = Point3D(1.25, 0, 0)
l1 = Edge3D(len,r1,r2,(r2-r1)/len)

Edge3D(1.0, [1.0, 0.0, 0.0], [2.0, 0.0, 0.0], [1.0, 0.0, 0.0])

In [14]:
l1(s)

2-element Vector{Float64}:
 0.75
 0.25

## Section 3: Integrals 

### Section 3.1: Point Interactions 

In [44]:
function f1(h,eta)
    return 1/eta^2*(sqrt(h^2+eta^2)-h)
end 

function f2(h,eta)
    return sqrt(h^2+eta^2)/(2*eta^2)-h^2/(2*eta^3)*asinh(eta/h)
end 

function f3(h,eta)
    return (2*h^3+(h^2+eta^2)^(3/2)-3*h^2*sqrt(h^2+eta^2))/(3*eta^4)
end 

function I8integrand(s,h,R0)
    arg = sqrt(R0^2+s^2)
    return f1(h,arg) 
end

function I8(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return I14(s,h,R0) + asinh(s/t0) - h/R0*atan(s/R0)
end

function I9(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return sqrt(t0^2+s^2)/h-asinh(h/sqrt(R0^2+s^2))-.5*log(R0^2+s^2) 
end

function I9integrand(s,h,R0)
    arg = sqrt(R0^2+s^2)
    return s/h*f1(h,arg) 
end

function I10(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return t0^2/(2*R0^2)*asinh(s/t0)-(h^2*s)/(2*R0^2*sqrt(R0^2+s^2))*asinh(sqrt(R0^2+s^2)/h)
end

function I10integrand(s,h,R0)
    arg = sqrt(R0^2+s^2)
    return f2(h,arg) 
end

function I11(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return sqrt(t0^2+s^2)/(2*h)+h*asinh(sqrt(R0^2+s^2)/h)/(2*sqrt(R0^2+s^2))
end

function I11integrand(s,h,R0)
    arg = sqrt(R0^2+s^2)
    return s/h*f2(h,arg) 
end

function I12(s,h,R0) 
    return I14(s,h,R0)/3+I15(s,h,R0)/3+I18(s,h,R0)+2*I19(s,h,R0)/3 
end

function I12integrand(s,h,R0)
    arg = sqrt(R0^2+s^2)
    return f3(h,arg) 
end

function I13(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return ((t0^2+s^2)^(3/2)/h-h^2)/(3*(R0^2+s^2))
end

function I13integrand(s,h,R0)
    arg = sqrt(R0^2+s^2)
    return s/h*f3(h,arg) 
end

function I14(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    argplus = sqrt(t0^2+s^2)+h
    argmin  = sqrt(t0^2+s^2)-h
    return sign(s)*(I17plus(argmin,h,R0) - I17min(argplus,h,R0)) 
end

function I15(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return asinh(s/t0)
end

function I16(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return h/R0*atan(s/R0)
end

function I17plus(x,h,R0)
    t0 = sqrt(h^2+R0^2)
    arg = -R0^2/(x*t0)+h/t0
    return h/(2*R0)*asin(arg) 
end 

function I17min(x,h,R0)
    t0 = sqrt(h^2+R0^2)
    arg  = -R0^2/(x*t0)-h/t0
    return h/(2*R0)*asin(arg)  
end

function I18(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return h^3/(3*R0^3)*(R0*s/(R0^2+s^2)+atan(s/R0)) 
end

function I19(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    value = -(h^2*s*sqrt(t0^2+s^2))/(2*R0^2*(R0^2+s^2))
    value += h/(2*R0)*(1-h^2/R0^2)*atan(h*s/(R0*sqrt(t0^2+s^2)))
    return value-I14(s,h,R0)
end;


### Section 3.2: Edge-Edge Interactions 

In [59]:
function I3(a::Point3D,el::Edge3D)
    num  = norm(el.r2-a) + norm(el.r2-a) + norm(el.r2-el.r1)
    den  = norm(el.r2-a) + norm(el.r2-a) - norm(el.r2-el.r1)
    return num / den 
end 

function Rphi1(a::Point3D,el::Edge3D)
    t1 = dot(el.r2-a,el.dir)/norm(el.r2-el.r1)*I3(a,el)
    t2 = (norm(el.r1-a) - norm(el.r2-a)) /norm(el.r2-el.r1)
    return t1+t2 
end 

function I25(s::Point3D,el1::Edge3D,el2::Edge3D)
    t1 = dot(el2.r2-s,el2.dir)/el1.len*I3(el1.r2,el2) - Rphi1(el1.r2,el2)
    t1 = t1*norm(el1.r2-s)^2/norm(el1.r2-el1.r1)
    t2 = dot(el2.r2-s,el2.dir)/el2.len*I3(el1.r1,el2) - Rphi1(el1.r1,el2)
    t2 = t2*norm(el1.r1-s)^2/norm(el1.r2-el1.r1)
    t3 = dot(el1.r2-s,el1.dir)/el1.len*I3(el2.r2,el1) - Rphi1(el2.r2,el1)
    t3 = t3*norm(el2.r2-s)^2/norm(el1.r2-el1.r1)
    t4 = dot(el1.r2-s,el1.dir)/el1.len*I3(el2.r1,el1) - Rphi1(el2.r1,el1)
    t4 = t4*norm(el2.r1-s)^2/norm(el1.r2-el1.r1)
    return t1-t2+t3-t4
end

function I26(s::Point3D,el1::Edge3D,el2::Edge3D)
    t1 = norm(el2.r2-s)^2/(2*el2.len)*I3(el1.r2,el2)
    t2 = norm(el1.r2-s)^2/(2*el2.len)*I3(el1.r1,el2)
    t3 = dot(el1.r2-s,el1.dir)/2*(dot(el2.r2-s,el2.dir)/el1.len*I3(el1.r2,el2)-Rphi1(el1.r2,el2))
    t4 = dot(el1.r1-s,el1.dir)/2*(dot(el2.r2-s,el2.dir)/el1.len*I3(el1.r1,el2)-Rphi1(el1.r1,el2)) 
    return t1-t2+t3-t4
end

function I27(s::Point3D,el1::Edge3D,el2::Edge3D)
    t1 = norm(el1.r2-s)^2/(2*el1.len)*I3(el1.r2,el2)
    t2 = norm(el1.r1-s)^2/(2*el1.len)*I3(el1.r1,el2)
    t3 = dot(el2.r2-s,el2.dir)/2*(dot(el1.r2-s,el1.dir)/el1.len*I3(el2.r2,el1)-Rphi1(el2.r2,el1))
    t4 = dot(el2.r1-s,el2.dir)/2*(dot(el1.r2-s,el1.dir)/el1.len*I3(el2.r1,el1)-Rphi1(el2.r1,el1))
    return t1-t2+t3-t4 
end

function I28(s::Point3D,el1::Edge3D,el2::Edge3D)
    t1 = dot(el1.r2-s,el1.dir)*I3(el1.r2,el2)
    t2 = dot(el1.r1-s,el1.dir)*I3(el1.r1,el2)
    t3 = dot(el2.r2-s,el2.dir)*I3(el2.r2,el1)
    t4 = dot(el2.r1-s,el2.dir)*I3(el2.r1,el1) 
    return t1-t2+t3-t4
end

function Qphi1theta1(s::Point3D,el1::Edge3D,el2::Edge3D)
    return I25(s,el1,el2)-el1(s)[1]*I26(s,el1,el2)-el2(s)[1]*I27(s,el1,el2)-el1(s)[1]*el2(s)[1]*I28(s,el1,el2)
end

Qphi1theta1 (generic function with 1 method)

### Testing 

In [29]:
r11 = Point3D(1.,0.,0.,); r12 = Point3D(2.,0.,0.); len = norm(r12-r11)
l1 = Edge3D(len,r11,r12,(r2-r1)/len)

r21 = Point3D(1.,0.,0.,); r22 = Point3D(2.,2.,0.); len = norm(r22-r21)
l2 = Edge3D(len,r21,r22,(r22-r21)/len)

s = Point3D(1.25, 0, 0)

3-element SVector{3, Float64} with indices SOneTo(3):
 1.25
 0.0
 0.0

In [41]:
l2(s)[1]

0.95

In [60]:
function Qphi1theta1(s::Point3D,el1::Edge3D,el2::Edge3D)
    return I25(s,el1,el2)
    #-el1(s)[1]*I26(s,el1,el2)-el2(s)[1]*I27(s,el1,el2)-el1(s)[1]*el2(s)[1]*I28(s,el1,el2)
end

Qphi1theta1 (generic function with 1 method)

In [61]:
Qphi1theta1(2*s,l1,l2)

3.89319701449895

In [62]:
I28(2*s,l1,l2)

7.35354607077237

## Section 4: Testing Function Evaluations

In [45]:
f3(.5,.5)

0.39052429175126946

## Section 5: Testing the Integrals

In [46]:
h = .5
R0 = .5
s1 = .1; s2 = 1. 
display(I8(s2,h,R0) - I8(s1,h,R0))
display(I9(s2,h,R0) - I9(s1,h,R0))
display(I10(s2,h,R0) - I10(s1,h,R0))
display(I11(s2,h,R0) - I11(s1,h,R0))
display(I12(s2,h,R0) - I12(s1,h,R0))
display(I13(s2,h,R0) - I13(s1,h,R0))

0.6411043239441898

0.6701632668649524

0.40233918997691

0.41699852046271757

0.29066193043886757

0.2997629487653599

In [47]:
display(hcubature(s -> I8integrand(s[1],h,R0) , (s1,), (s2,)))
display(hcubature(s -> I9integrand(s[1],h,R0) , (s1,), (s2,)))
display(hcubature(s -> I10integrand(s[1],h,R0) , (s1,), (s2,)))
display(hcubature(s -> I11integrand(s[1],h,R0) , (s1,), (s2,)))
display(hcubature(s -> I12integrand(s[1],h,R0) , (s1,), (s2,)))
display(hcubature(s -> I13integrand(s[1],h,R0) , (s1,), (s2,)))

(0.64110432394419, 7.382316979942516e-11)

(0.6701632668649526, 2.050949410303815e-10)

(0.40233918997690965, 7.061135010033581e-11)

(0.41699852046271785, 1.7793388984443936e-10)

(0.29066193043886746, 6.711931010983108e-11)

(0.2997629487653601, 1.5622747540078308e-10)